Header 123

In [2]:
!pip install datasets
!pip install youtube-comment-downloader
!pip install youtube-search-python
!pip install "httpx<0.24.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 15.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is

In [ ]:
from datasets import load_dataset, Dataset, concatenate_datasets
import json
import pandas as pd

# === 1. Amazon Reviews (Meinungen, sehr gut für Sentiment) ===
# mlsum_all = load_dataset("mlsum", "de", split="train")
# mlsum_text = mlsum_all.map(lambda x: {"text": x["text"]}, remove_columns=mlsum_all.column_names)

# === 2. OpenSubtitles (Film-Dialoge, implizites Sentiment) ===
# subtitles = load_dataset("opus_books", "de-en", split="train[:20000]")
# subtitles = subtitles.map(lambda x: {"text": x["translation"]["de"]}, remove_columns=subtitles.column_names)


# === 3. SB10k (from Hugging Face) ===
sb10k = load_dataset("Alienmaster/SB10k", split="train")
print(sb10k.column_names)
sb10k = sb10k.map(lambda x: {
    "text": x["Text"],
    "label": x["Sentiment"].lower()  # Convert to 'positive', 'neutral', 'negative'
}, remove_columns=sb10k.column_names)

files.download(sb10k)

# === Kombinieren & Shuffle ===
combined_dataset = concatenate_datasets([sb10k])
combined_dataset = combined_dataset.shuffle(seed=42)

# === Speichern als JSONL ===
with open("german_sentiment_dataset.jsonl", "w", encoding="utf-8") as f:
    for example in combined_dataset:
        json.dump(example, f, ensure_ascii=False)
        f.write("\n")

print(f"✅ Fertig: {len(combined_dataset)} Texte gespeichert.")

['ID', 'Sentiment', 'Text', 'Normalized', 'POS-Tags', 'Dependency Labels', 'additional Annotations']


TypeError: stat: path should be string, bytes, os.PathLike or integer, not Dataset

In [ ]:
from datasets import load_dataset, Dataset, concatenate_datasets
import json
import pandas as pd
from google.colab import files

def print_dataset_preview(dataset, name=""):
    print(f"\n📄 Preview of {name} (first 5 samples):")
    for i in range(min(20, len(dataset))):
        example = dataset[i]
        text = example.get("text", "[No text field]")
        label = example.get("label", "[No label]")
        print(f"{i+1}. Text: {text[:150]}")  # Limit to first 150 chars
        print(f"   Label: {label}\n")

def save_dataset_preview(dataset, name="dataset", max_samples=20):
    filename = f"{name}_preview.txt"
    with open(filename, "w", encoding="utf-8") as f:
        f.write(f"📄 Preview of {name} (first {max_samples} samples):\n\n")
        for i in range(min(max_samples, len(dataset))):
            example = dataset[i]
            text = example.get("text", "[No text field]")
            label = example.get("label", "[No label]")
            f.write(f"{i+1}. Text: {text[:150]}\n")
            f.write(f"   Label: {label}\n\n")
    return filename

def save_full_dataset(dataset, name="dataset"):
    filename = f"{name}.jsonl"
    with open(filename, "w", encoding="utf-8") as f:
        for example in dataset:
            text = example.get("text", None)
            label = example.get("label", None)
            json.dump({"text": text, "label": label}, f, ensure_ascii=False)
            f.write("\n")
    return filename  # ✅ return the path

print_dataset_preview(sb10k, name="SB10k")

file1 = save_full_dataset(sb10k, name="SB10k")

# Download all files
files.download(file1)


📄 Preview of SB10k (first 5 samples):
1. Text: RT @TheKedosZone : So ein Hearthstone - Key von @BlizzardCSEU_EN für den netten YouTube - Onkel Kedos wäre auch schon was ganz feines . * hust *
   Label: positive

2. Text: Tainted Talents ( Ateliertagebuch. ) " Wir sind nicht allein http://t.co/9rhta65MSx
   Label: neutral

3. Text: Aber wenigstens kommt #Supernatural heute mal wieder um 22  Uhr - ein schwacher Trost
   Label: neutral

4. Text: DARLEHEN - Angebot für Schufa - freie Darlehen : Günstiger Anbieter für Online DARLEHEN in Deutschland http://t.co/G84qcIGk7k
   Label: neutral

5. Text: ANRUF ERWÜNSCHT : Hardcore Teeny Vicky Carrera : Hallo mein süsser , ich bin die 20  jährige Vicky aus Deutschland .... http://t.co/LvwyZgew4Q
   Label: neutral

6. Text: Na ? Wo sind Frankens heimliche Talente ? - Die ersten Bewerbungen für den Talentwettbewerb " Wer ko , der darf ! " ... http://t.co/MHGsyZDOdJ
   Label: neutral

7. Text: ... Glück breitet sich aus ...
   Label: positive

8. Te

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
import pandas as pd

# This will prompt you to upload the file manually
uploaded = files.upload()


# Replace with actual file name (should match uploaded file)
df = pd.read_csv("filmstarts.tsv", sep="\t", quoting=3, on_bad_lines='skip', encoding="utf-8", engine="python")

# Preview the first rows
print(df.head())


Saving filmstarts.tsv to filmstarts (1).tsv
   http://www.filmstarts.de/kritiken/27070.html    5  \
0  http://www.filmstarts.de/kritiken/27070.html  5.0   
1  http://www.filmstarts.de/kritiken/27070.html  5.0   
2  http://www.filmstarts.de/kritiken/27070.html  5.0   
3  http://www.filmstarts.de/kritiken/27070.html  5.0   
4  http://www.filmstarts.de/kritiken/27070.html  5.0   

  Der Herr der Ringe - Die Gefährten ist für mich der beste Teil der Mittelerde Saga. Der Beginn der Geschichte im Auenland, die Verfolgung durch die Nazgul und die Flucht nach Bree, die Wetterspitze, Bruchtal, Elronds Rat, die Minen von Moria, die Brücke von Khazad Dum, Lothlorien und zum Abschluss die Trennung der Gefährten. All das bietet ein spannendes und abwechslungsreiches Abenteuer in einer perfekten Spielfilmlänge. Auch die Luftbildaufnahmen und Effekte sind in Teil 1 der Trilogie vollends überzeugend! Die perfekt abgestimmte Filmmusik zu den versch Orten in Mittelerde rundet das Spektakel ab. Von mir g

In [ ]:
from youtube_comment_downloader import YoutubeCommentDownloader
import json

def scrape_youtube_comments(video_url, max_count=1000, output_file="youtube_comments.jsonl"):
    downloader = YoutubeCommentDownloader()
    comments = downloader.get_comments_from_url(video_url, sort_by=0)  # 0 = Top, 1 = Newest

    with open(output_file, "w", encoding="utf-8") as f:
        for i, comment in enumerate(comments):
            if i >= max_count:
                break
            text = comment.get("text", "").strip()
            if text:
                json.dump({"text": text}, f, ensure_ascii=False)
                f.write("\n")

    print(f"✅ Saved {min(i+1, max_count)} comments to {output_file}")

# Print first 10 lines of the JSONL file
with open("youtube_comments.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i >= 9:
            break


{"text": "can confirm: he never gave us up"}
{"text": "If this doesn't teach you not to click on a dark web link, nothing will."}
{"text": "Wait this isn’t How To Rickroll People Tutorial."}
{"text": "My professor sent us this link as the \"final exam key\"........"}
{"text": "It’s come to the point that being rickrolled is a privilege"}
{"text": "Wait, this isn’t the link to  finding out Obama’s last name"}
{"text": "Where is my free minecraft account with 1 billion minecoins"}
{"text": "I just got rick rolled by scanning someone’s hairline"}
{"text": "this isn't over until all 7 billion people in the world are rick rolled - 1/7th there ❤️"}
{"text": "Is everyone going to ignore the fact that Rick Astley looks like a twelve year old boy but has a uniquely deep voice?"}


In [ ]:
video_url = "https://www.youtube.com/watch?v=dQw4w9WgXcQ"  # Replace with your own
scrape_youtube_comments(video_url, max_count=500)

print()

✅ Saved 500 comments to youtube_comments.jsonl



In [4]:
from youtubesearchpython import VideosSearch
from youtube_comment_downloader import YoutubeCommentDownloader
import json
import os
import time

# === Config ===
domains = {
    "gaming": ["gaming deutsch", "spiel bewertung", "pc spiele test", "konsole vergleich deutsch"],
    "music": ["deutschrap 2024", "musikvideo reaktion", "musik bewertung"],
    "products": ["produkttest deutsch", "smartphone review deutsch", "technik empfehlung"],
    "politics": ["nachrichten meinung deutsch", "politische debatte deutschland", "bundestag diskussion"],
    "news": ["tagesschau aktuell", "zdf heute meinung", "nachrichten kommentar"],
    "travel": ["hotel erfahrung deutsch", "reisebericht deutschland", "urlaub bewertung"]
}

videos_per_query = 5
max_comments_per_video = 1000
output_file = "youtube_comments_200MB.jsonl"
target_size_MB = 1

# === Init ===
downloader = YoutubeCommentDownloader()
total_comments = 0

def file_size_mb(path):
    return os.path.getsize(path) / (1024 * 1024)

# === Open output file ===
with open(output_file, "w", encoding="utf-8") as out_file:
    for domain, query_list in domains.items():
        print(f"\n🔍 DOMAIN: {domain.upper()}")

        for query in query_list:
            print(f"🔸 Searching: {query}")
            try:
                videos_search = VideosSearch(query, limit=videos_per_query)
                results = videos_search.result()["result"]
            except Exception as e:
                print(f"⚠️ Failed to search videos for '{query}': {e}")
                continue

            for i, video in enumerate(results):
                url = video["link"]
                print(f"  {i+1}. {video['title']}\n     URL: {url}")
                try:
                    count = 0
                    for comment in downloader.get_comments_from_url(url, sort_by=0):
                        text = comment.get("text", "").strip()
                        if text:
                            json.dump({"text": text, "domain": domain, "query": query}, out_file, ensure_ascii=False)
                            out_file.write("\n")
                            count += 1
                            total_comments += 1

                            # Stop collecting from this video
                            if count >= max_comments_per_video:
                                break

                            # Stop if total file size is big enough
                            if file_size_mb(output_file) >= target_size_MB:
                                print(f"\n✅ Stopped: Reached {target_size_MB}MB after {total_comments} comments.")
                                raise StopIteration

                    print(f"     ✔ Collected {count} comments.")

                except StopIteration:
                    raise
                except Exception as e:
                    print(f"     ⚠️ Error while downloading comments: {e}")

                time.sleep(1)

try:
    print(f"\n🎉 Done! Total comments collected: {total_comments}")
except:
    pass


🔍 DOMAIN: GAMING
🔸 Searching: gaming deutsch
  1. Wir sind ein Eichhörnchen mit Waffe... (Das Game geht unironisch hart)
     URL: https://www.youtube.com/watch?v=5DnJbnMhXNM
     ✔ Collected 547 comments.
  2. Wir spielen den Breaking Bad Simulator...
     URL: https://www.youtube.com/watch?v=6lT-eXOKnH4
     ✔ Collected 1000 comments.
  3. Dieses Horrorspiel gibt mir einen Herzinfarkt...
     URL: https://www.youtube.com/watch?v=C5dsZZGVzsM
     ✔ Collected 626 comments.
  4. Das LUSTIGSTE Game seit Lethal Company!
     URL: https://www.youtube.com/watch?v=fIo4o7YyFqI
     ✔ Collected 306 comments.
  5. Endlich habe ich es fertig geterraformed... #MorizzGMC #minecraft
     URL: https://www.youtube.com/watch?v=YeXu47loaQQ
     ✔ Collected 16 comments.
🔸 Searching: spiel bewertung
  1. Spiel doch mal SKY TEAM! - Brettspiel Rezension Meinung Test #479
     URL: https://www.youtube.com/watch?v=XYhDahKM9h8
     ✔ Collected 17 comments.
  2. 💎 WWE 2K25 - TEST | Das beste WWE-Spiel seit Ja

StopIteration: 

In [ ]:
import sys
print(sys.executable)

C:\Users\marku\miniconda3\envs\ml\python.exe


In [ ]:
!{sys.executable} -m pip install matplotlib

   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ------ --------------------------------- 1.3/8.1 MB 12.4 MB/s eta 0:00:01
   ----------- ---------------------------- 2.4/8.1 MB 6.3 MB/s eta 0:00:01
   --------------- ------------------------ 3.1/8.1 MB 5.4 MB/s eta 0:00:01
   -------------------- ------------------- 4.2/8.1 MB 4.9 MB/s eta 0:00:01
   ----------------------- ---------------- 4.7/8.1 MB 4.5 MB/s eta 0:00:01
   --------------------------- ------------ 5.5/8.1 MB 4.2 MB/s eta 0:00:01
   ------------------------------- -------- 6.3/8.1 MB 4.2 MB/s eta 0:00:01
   ------------------------------------ --- 7.3/8.1 MB 4.3 MB/s eta 0:00:01
   ---------------------------------------- 8.1/8.1 MB 4.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   -------------- ------------------------- 0.8/2.2 MB 3.8 MB/s eta 0:00:01
   ---------------------------- ----------- 1.6/2.2 MB 4.0 MB/s eta 0:00:01
   ----------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
